# 02 · Calendar signal checks

**Purpose:** verify that the Calendar rebalancing signal in
`src/rebalancing_frontrun/signals/calendar.py` behaves as intended before any backtest uses it.

**What the signal is:** a simulated 60/40 ES/ZN portfolio drifts with daily returns and is
reset to 60/40 after the close of the last trading day of each month. The signal is the
equity weight minus 60%, measured at each close *before* that day's reset.

- `signal > 0` → equities overweight → rebalancers sell ES, buy ZN
- `signal < 0` → equities underweight → rebalancers buy ES, sell ZN

**Checks in this notebook**
1. Signal over time, with month-ends highlighted
2. Signal vs. the paper's rule of thumb: 0.24 × (month-to-date ES return − ZN return)
3. Month-end calendar checks (holidays, partial first and last months)
4. Largest month-end signals and the market events behind them

**Inputs:** `data/processed/returns.parquet` (built by `data/settlements.py`)

## Setup

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

from rebalancing_frontrun.data.databento_loader import REPO_ROOT
from rebalancing_frontrun.signals.calendar import TARGET, calendar_signal, trading_calendar

returns = pd.read_parquet(REPO_ROOT / "data" / "processed" / "returns.parquet")

# One frame with returns, calendar flags and the signal
df = returns.join(trading_calendar(returns.index))
df["signal"] = calendar_signal(returns)

print(f"{len(df):,} trading days, {df.index.min():%Y-%m-%d} -> {df.index.max():%Y-%m-%d}")
print(f"month-ends: {df['is_month_end'].sum()}")
df[["es_ret", "zn_ret", "days_to_month_end", "is_month_end", "signal"]].head()

2,752 trading days, 2015-11-20 -> 2026-10-26
month-ends: 131


,es_ret,zn_ret,days_to_month_end,is_month_end,signal
trade_date,,,,,
2015-11-20,0.004569,-0.000863,5.0,False,0.001300
2015-11-23,-0.002154,0.001110,4.0,False,0.000517
2015-11-24,0.000240,0.000739,3.0,False,0.000397
2015-11-25,0.001559,0.000616,2.0,False,0.000623
2015-11-27,0.000958,0.000738,1.0,False,0.000676


## 1 · Signal over time

The signal should build up through each month and snap back to ~0 at the start of the next
(the portfolio resets after each month-end close). Red dots mark the month-end values, which
are the drifts rebalancers actually have to trade.

In [2]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df["signal"] * 100, lw=0.8, color="0.4", label="daily signal")
me = df[df["is_month_end"]]
ax.scatter(me.index, me["signal"] * 100, s=10, color="tab:red", zorder=3, label="month-end")
ax.axhline(0, color="black", lw=0.6)
ax.set_ylabel("equity weight − 60% (pct pts)")
ax.set_title("Calendar signal: 60/40 drift, reset each month-end")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

**Check:** the day after each month-end, the signal should be small, roughly 0.24 × one
day's excess return, because the portfolio just reset.

In [3]:
after_reset = df["signal"].shift(-1)[df["is_month_end"]].dropna()
approx_one_day = 0.24 * (df["es_ret"] - df["zn_ret"]).fillna(0).shift(-1)[df["is_month_end"]].dropna()
print("day-after-reset signal, abs mean (pct pts):", round(after_reset.abs().mean() * 100, 3))
print("max abs diff vs 0.24 x one-day excess (pct pts):",
      round((after_reset - approx_one_day).abs().max() * 100, 4))

day-after-reset signal, abs mean (pct pts): 0.196
max abs diff vs 0.24 x one-day excess (pct pts): 0.0337


## 2 · Signal vs. rule of thumb

Because 0.6 × 0.4 = 0.24, the drift should be close to 0.24 × (month-to-date ES return −
month-to-date ZN return). The match is not exact (compounding, and weights shift as they
drift), but the correlation should be very close to 1.

In [4]:
month = df.index.to_period("M")
es_mtd = (1 + df["es_ret"].fillna(0)).groupby(month).cumprod() - 1
zn_mtd = (1 + df["zn_ret"].fillna(0)).groupby(month).cumprod() - 1
df["rule_of_thumb"] = TARGET * (1 - TARGET) * (es_mtd - zn_mtd)

me = df[df["is_month_end"]]
corr = me["signal"].corr(me["rule_of_thumb"])
print(f"month-end correlation: {corr:.4f}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(me["rule_of_thumb"] * 100, me["signal"] * 100, s=12)
lim = max(me[["signal", "rule_of_thumb"]].abs().max()) * 100 * 1.1
ax.plot([-lim, lim], [-lim, lim], color="black", lw=0.6, ls="--")
ax.set_xlabel("0.24 × MTD (ES − ZN) return (pct pts)")
ax.set_ylabel("simulated signal (pct pts)")
ax.set_title(f"Month-end signal vs rule of thumb (corr {corr:.3f})")
plt.tight_layout()
plt.show()

month-end correlation: 0.9992


## 3 · Calendar checks

Month-ends come from the actual trading days in the data, so holidays are handled
automatically. Things to confirm:
- every complete month has exactly one month-end,
- month-ends that fall on or near holidays land on the right day,
- the partial first month (data starts 2015-11-20) and the incomplete last month
  (data stops before October 2026 ends) are handled.

In [5]:
months = df.index.to_period("M")
per_month = df.groupby(months)["is_month_end"].sum()
print("months with != 1 month-end:")
print(per_month[per_month != 1])

months with != 1 month-end:
trade_date
2026-10    0
Freq: M, Name: is_month_end, dtype: int64


In [6]:
# Month-ends that are NOT the last weekday of the month (holidays, e.g. Good Friday)
me_dates = df.index[df["is_month_end"]]
last_weekday = me_dates + pd.offsets.BMonthEnd(0)
holiday_shifted = pd.DataFrame({"month_end_used": me_dates, "last_weekday": last_weekday})
holiday_shifted[holiday_shifted["month_end_used"] != holiday_shifted["last_weekday"]]

,month_end_used,last_weekday
28,2018-03-29,2018-03-30
66,2021-05-28,2021-05-31
100,2024-03-28,2024-03-29


In [7]:
# Partial first month and incomplete last month
print("first month:")
print(df[["days_to_month_end", "is_month_end", "signal"]].head(3))
print("\nlast month (days_to_month_end should be NaN):")
print(df[["days_to_month_end", "is_month_end", "signal"]].tail(3))

first month:
            days_to_month_end  is_month_end    signal
trade_date                                           
2015-11-20                5.0         False  0.001300
2015-11-23                4.0         False  0.000517
2015-11-24                3.0         False  0.000397

last month (days_to_month_end should be NaN):
            days_to_month_end  is_month_end    signal
trade_date                                           
2026-10-22                NaN         False -0.005376
2026-10-23                NaN         False -0.006213
2026-10-26                NaN         False -0.004390


**Note:** the first month (Nov 2015) starts on the 20th, so its drift covers only part of
the month. The strategy only trades the last 5 days, and Nov 2015's last 5 days are in the
data, but its month-end signal reflects ~7 trading days of drift rather than a full month.

## 4 · Largest month-end signals

These should line up with big stock-vs-bond moves. Large negative = equities sold off
(pensions must buy stocks); large positive = equities rallied (pensions must sell stocks).

In [8]:
me = df[df["is_month_end"]].copy()
me["es_mtd"], me["zn_mtd"] = es_mtd[me.index], zn_mtd[me.index]
cols = ["signal", "es_mtd", "zn_mtd"]
print("Most negative (equities underweight):")
print((me.nsmallest(5, "signal")[cols] * 100).round(2))
print("\nMost positive (equities overweight):")
print((me.nlargest(5, "signal")[cols] * 100).round(2))

Most negative (equities underweight):
            signal  es_mtd  zn_mtd
trade_date                        
2020-03-31   -3.96  -12.52    2.92
2018-12-31   -2.88   -9.29    2.15
2020-02-28   -2.73   -8.47    2.42
2019-05-31   -2.20   -6.65    2.22
2016-01-29   -1.98   -5.18    2.92

Most positive (equities overweight):
            signal  es_mtd  zn_mtd
trade_date                        
2020-04-30    2.82   12.95    0.27
2020-11-30    2.42   10.98    0.23
2026-04-30    2.41   10.24   -0.41
2022-10-31    2.10    7.82   -1.31
2023-06-30    1.85    6.00   -1.92


## Summary

Fill in after running:
- [ ] Signal resets each month (section 1)
- [ ] Month-end correlation with rule of thumb ≈ 1 (section 2)
- [ ] One month-end per complete month; holiday months correct; last month NaN (section 3)
- [ ] Largest signals match known market events (section 4)

Next: the strategy itself (positions in the last 5 trading days, sized from the prior close's
signal), in `backtest/`.